# Give Me Credit — Exploratory Data Analysis

This notebook documents the key data quality characteristics of the Give Me Credit dataset before any training code is written. Every preprocessing decision made in the training pipeline traces back to a finding documented here.

**Dataset:** Kaggle Give Me Some Credit (cs-training.csv)  
**Rows:** 150,000  **Features:** 11  

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'axes.titlesize': 14, 'axes.labelsize': 12})

print('pandas', pd.__version__)
print('seaborn', sns.__version__)

## Section 1 — Dataset Overview

In [ ]:
df = pd.read_csv('../data/raw/cs-training.csv', index_col=0)

print('Shape:', df.shape)
assert df.shape == (150000, 11), f'Expected (150000, 11), got {df.shape}'

In [ ]:
print('Column dtypes:')
print(df.dtypes)

In [ ]:
df.head()

## Section 2 — Target Distribution

In [ ]:
target_counts = df['SeriousDlqin2yrs'].value_counts().sort_index()
default_rate = df['SeriousDlqin2yrs'].mean() * 100

print('Target value counts:')
print(target_counts)
print(f'\nDefault rate: {default_rate:.2f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(
    ['Non-default (0)', 'Default (1)'],
    target_counts.values,
    color=['steelblue', 'tomato'],
    width=0.5
)

for bar, count in zip(bars, target_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 500,
        f'{count:,}',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax.set_title('Target Distribution — SeriousDlqin2yrs', fontsize=14)
ax.set_ylabel('Count')
ax.set_xlabel('Class')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()
print(f'Default rate: {default_rate:.2f}%')

The 7% default rate means the training set has ~10,500 positives in 150,000 rows. This severe imbalance requires SMOTE oversampling — without correction, a classifier predicting all zeros achieves 93% accuracy while being useless.

## Section 3 — Missing Value Profile

In [ ]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'null_count': null_counts,
    'null_pct': null_pct
}).sort_values('null_pct', ascending=False)

print('Missing value profile (sorted by % missing):')
print(missing_df[missing_df['null_count'] > 0].to_string())

In [ ]:
# Only plot columns with at least some missing values
missing_nonzero = missing_df[missing_df['null_pct'] > 0]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(
    missing_nonzero.index,
    missing_nonzero['null_pct'],
    color='salmon'
)

for bar, pct in zip(bars, missing_nonzero['null_pct']):
    ax.text(
        bar.get_width() + 0.1,
        bar.get_y() + bar.get_height() / 2,
        f'{pct:.1f}%',
        va='center', fontsize=11
    )

ax.set_title('Missing Value Profile (% NaN per column)', fontsize=14)
ax.set_xlabel('% Missing')
ax.set_ylabel('Column')
ax.set_xlim(0, missing_nonzero['null_pct'].max() * 1.2)
plt.tight_layout()
plt.show()

**Missing value decisions:**

- `MonthlyIncome` is missing for ~19.8% of rows. The distribution is heavily right-skewed (a small number of very high earners pull the mean upward), so the mean is not a representative central value. **Median imputation** is used — fitted only on the training split to prevent leakage from the test set.

- A binary flag column `MonthlyIncome_was_missing` will be added alongside the imputed values. Missingness may correlate with income level (e.g., self-employed, unemployed) and preserving this signal as a feature lets the model learn from it rather than discarding it.

- `NumberOfDependents` is missing for ~2.6% of rows. Median imputation is applied here as well (integer feature, low missing rate, median = mode = 0 in this dataset).

## Section 4 — Feature Distributions

In [ ]:
feature_cols = [c for c in df.columns if c != 'SeriousDlqin2yrs']
log_scale_cols = {'RevolvingUtilizationOfUnsecuredLines', 'DebtRatio'}

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    data = df[col].dropna()
    ax.hist(data, bins=50, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(col, fontsize=12, wrap=True)
    ax.set_xlabel('Value', fontsize=9)
    ax.set_ylabel('Count', fontsize=9)
    ax.tick_params(labelsize=8)
    if col in log_scale_cols:
        ax.set_yscale('log')
        ax.set_title(col + ' (log y)', fontsize=12)

fig.suptitle('Feature Distributions — Give Me Credit', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## Section 5 — Outlier Investigation

In [ ]:
outlier_cols = ['NumberOfTimes90DaysLate', 'NumberOfTime30-59DaysPastDueNotWorse']

for col in outlier_cols:
    print(f'--- value_counts for {col} (top 20) ---')
    print(df[col].value_counts().sort_index(ascending=False).head(20))
    print()

Values of 96 and 98 appear ~220 times in `NumberOfTimes90DaysLate`. These are likely data entry errors (96/98 may be sentinel codes). Multiple Kaggle solutions cap at 17 (the next plausible maximum given quarterly credit reporting cycles). We apply `clip(upper=17)`.

In [ ]:
col = 'NumberOfTimes90DaysLate'

print('Before clip — tail of value_counts:')
print(df[col].value_counts().sort_index(ascending=False).head(10))

# Apply cap (non-destructive preview)
capped = df[col].clip(upper=17)

print('\nAfter clip(upper=17) — tail of value_counts:')
print(capped.value_counts().sort_index(ascending=False).head(10))

n_affected = (df[col] > 17).sum()
print(f'\nRows affected by clip: {n_affected}')

In [ ]:
# Same check for NumberOfTime30-59DaysPastDueNotWorse
col2 = 'NumberOfTime30-59DaysPastDueNotWorse'

print('Before clip — tail of value_counts (NumberOfTime30-59DaysPastDueNotWorse):')
print(df[col2].value_counts().sort_index(ascending=False).head(10))

capped2 = df[col2].clip(upper=17)
n_affected2 = (df[col2] > 17).sum()
print(f'\nRows affected by clip(upper=17): {n_affected2}')

## Section 6 — Spearman Correlation Heatmap

Spearman rank correlation is used instead of Pearson because credit features are heavily right-skewed. Pearson assumes linear relationships and is sensitive to outliers — both assumptions are violated here.

In [ ]:
# Drop NaN rows before computing correlation
df_complete = df.dropna()
print(f'Rows used for correlation (after dropna): {len(df_complete):,} of {len(df):,}')

spearman_corr = df_complete.corr(method='spearman')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

mask = np.zeros_like(spearman_corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True  # upper triangle

sns.heatmap(
    spearman_corr,
    annot=True,
    fmt='.2f',
    cmap='RdYlBu_r',
    center=0,
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 8}
)

ax.set_title('Spearman Rank Correlation — Give Me Credit', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Identify the strongest pairwise correlations (excluding self-correlation)
corr_pairs = (
    spearman_corr.where(np.tril(np.ones(spearman_corr.shape), k=-1).astype(bool))
    .stack()
    .abs()
    .sort_values(ascending=False)
    .head(10)
)

print('Top 10 strongest Spearman correlations (absolute value):')
print(corr_pairs.round(3).to_string())

**Key observation:** The three late-payment count features — `NumberOfTime30-59DaysPastDueNotWorse`, `NumberOfTime60-89DaysPastDueNotWorse`, and `NumberOfTimes90DaysLate` — are very highly correlated with each other (~0.98). This makes intuitive sense: borrowers who miss 90-day payments almost always also have 30-day and 60-day misses on record. Despite the multicollinearity, all three are retained since tree-based models (LightGBM) handle correlated features well and each feature may capture a slightly different default timing signal.

## Section 7 — Class Imbalance Summary and SMOTE Rationale

**Summary: Give Me Credit has three preprocessing challenges that must be addressed before training:**

1. **7% class imbalance** — The dataset contains ~10,500 defaults out of 150,000 rows. A naive classifier predicting all non-default achieves 93% accuracy but zero recall on defaults. This is addressed with SMOTE (Synthetic Minority Oversampling Technique) applied **only to the training split**.

2. **MonthlyIncome 19.8% NaN** — Addressed with median imputation fitted only on the training fold (to prevent leakage), plus a binary `MonthlyIncome_was_missing` flag column added to preserve the missingness signal. The right-skewed distribution of income makes median the appropriate central tendency measure.

3. **NumberOfTimes90DaysLate outliers at 96/98** — These values are almost certainly data entry errors or sentinel codes, not genuine delinquency counts. Addressed with `clip(upper=17)` — the same cap applied by the top Kaggle solutions. The same logic applies to `NumberOfTime30-59DaysPastDueNotWorse`.

**SMOTE application order is critical:** SMOTE is applied **AFTER** the train/test split — never before. Applying SMOTE before splitting would allow synthetic samples generated from test-set minority instances to appear in the training set, causing information leakage and inflated evaluation metrics. This is the only correct approach per imbalanced-learn official documentation.

In [ ]:
# Final summary statistics confirming all findings
print('=== EDA Summary ===')
print(f'Dataset shape: {df.shape}')
print(f'Default rate: {df["SeriousDlqin2yrs"].mean() * 100:.2f}%')
print(f'MonthlyIncome NaN rate: {df["MonthlyIncome"].isnull().mean() * 100:.2f}%')
print(f'NumberOfDependents NaN rate: {df["NumberOfDependents"].isnull().mean() * 100:.2f}%')
print(f'NumberOfTimes90DaysLate > 17: {(df["NumberOfTimes90DaysLate"] > 17).sum()} rows')
print(f'NumberOfTime30-59 > 17: {(df["NumberOfTime30-59DaysPastDueNotWorse"] > 17).sum()} rows')